# ChatGPT code

In [1]:
from collections import deque


class Dinic:
    def __init__(self, n):
        self.n = n
        self.graph = [[] for _ in range(n)]

    def add_edge(self, u, v, capacity):
        # 正向边: [终点, 剩余容量, 反向边在邻接表中的下标]
        forward = [v, capacity, len(self.graph[v])]
        # 反向边: 初始容量为 0
        backward = [u, 0, len(self.graph[u])]

        self.graph[u].append(forward)
        self.graph[v].append(backward)

    def bfs(self, source, sink):
        """构建分层图"""
        self.level = [-1] * self.n
        self.level[source] = 0
        queue = deque([source])

        while queue:
            u = queue.popleft()

            for v, capacity, _ in self.graph[u]:
                if capacity > 0 and self.level[v] == -1:
                    self.level[v] = self.level[u] + 1
                    queue.append(v)

        return self.level[sink] != -1

    def dfs(self, u, sink, flow):
        """在分层图中寻找增广流"""
        if u == sink:
            return flow

        while self.ptr[u] < len(self.graph[u]):
            edge_index = self.ptr[u]
            v, capacity, reverse_index = self.graph[u][edge_index]

            if capacity > 0 and self.level[v] == self.level[u] + 1:
                pushed = self.dfs(v, sink, min(flow, capacity))

                if pushed > 0:
                    # 更新正向边剩余容量
                    self.graph[u][edge_index][1] -= pushed
                    # 更新反向边剩余容量
                    self.graph[v][reverse_index][1] += pushed
                    return pushed

            self.ptr[u] += 1

        return 0

    def max_flow(self, source, sink):
        total_flow = 0
        INF = float("inf")

        while self.bfs(source, sink):
            self.ptr = [0] * self.n

            while True:
                pushed = self.dfs(source, sink, INF)

                if pushed == 0:
                    break

                total_flow += pushed

        return total_flow


# 节点编号
# S = 0, A = 1, B = 2, T = 3
S, A, B, T = 0, 1, 2, 3

network = Dinic(4)

network.add_edge(S, A, 10)
network.add_edge(S, B, 8)
network.add_edge(A, B, 5)
network.add_edge(A, T, 7)
network.add_edge(B, T, 10)

answer = network.max_flow(S, T)

print(f"每天最大运输量为：{answer} 吨")

每天最大运输量为：17 吨


In [2]:
from collections import defaultdict


class FordFulkerson:
    def __init__(self):
        # residual[u][v] 表示 u -> v 的剩余容量
        self.residual = defaultdict(lambda: defaultdict(int))
        self.nodes = set()

    def add_edge(self, u, v, capacity):
        """添加一条容量为 capacity 的有向边 u -> v"""
        self.residual[u][v] += capacity
        self.nodes.add(u)
        self.nodes.add(v)

    def find_augmenting_path(self, source, sink):
        """
        用 DFS 在残量网络寻找一条增广路径。
        返回 path（边列表）和该路径的瓶颈容量。
        """
        visited = set()
        parent = {}

        def dfs(u):
            if u == sink:
                return True

            visited.add(u)

            for v, capacity in self.residual[u].items():
                if v not in visited and capacity > 0:
                    parent[v] = u

                    if dfs(v):
                        return True

            return False

        if not dfs(source):
            return None, 0

        # 反向回溯路径，并寻找瓶颈容量
        path = []
        bottleneck = float("inf")
        current = sink

        while current != source:
            previous = parent[current]
            path.append((previous, current))
            bottleneck = min(bottleneck, self.residual[previous][current])
            current = previous

        path.reverse()
        return path, bottleneck

    def max_flow(self, source, sink):
        max_flow_value = 0

        while True:
            path, bottleneck = self.find_augmenting_path(source, sink)

            # 找不到增广路径，说明已经达到最大流
            if bottleneck == 0:
                break

            print(f"增广路径: {' -> '.join(path_node for edge in path for path_node in [])}")
            print(f"本次增加流量: {bottleneck}")

            # 更新残量网络
            for u, v in path:
                self.residual[u][v] -= bottleneck
                self.residual[v][u] += bottleneck

            max_flow_value += bottleneck

        return max_flow_value


# ------------------ 使用示例 ------------------

network = FordFulkerson()

network.add_edge("S", "A", 10)
network.add_edge("S", "B", 8)
network.add_edge("A", "B", 5)
network.add_edge("A", "T", 7)
network.add_edge("B", "T", 10)

result = network.max_flow("S", "T")

print(f"\n最大流为: {result}")

增广路径: 
本次增加流量: 5
增广路径: 
本次增加流量: 5
增广路径: 
本次增加流量: 5
增广路径: 
本次增加流量: 2

最大流为: 17


In [3]:
from collections import defaultdict


def find_path(residual, source, sink):
    """DFS 寻找一条增广路径，返回节点路径；找不到则返回 None。"""
    visited = set()
    parent = {}
    stack = [source]
    visited.add(source)

    while stack:
        u = stack.pop()

        if u == sink:
            break

        for v, capacity in residual[u].items():
            if capacity > 0 and v not in visited:
                visited.add(v)
                parent[v] = u
                stack.append(v)

    if sink not in visited:
        return None

    path = []
    current = sink

    while current != source:
        path.append(current)
        current = parent[current]

    path.append(source)
    path.reverse()
    return path


def ford_fulkerson(edges, source, sink):
    """
    edges: [(起点, 终点, 容量), ...]
    返回最大流。
    """
    residual = defaultdict(lambda: defaultdict(int))
    max_flow = 0

    # 初始化残量网络
    for u, v, capacity in edges:
        residual[u][v] += capacity
        residual[v][u] += 0  # 确保反向边存在

    while True:
        path = find_path(residual, source, sink)

        if path is None:
            break

        # 计算该路径的瓶颈容量
        bottleneck = min(
            residual[path[i]][path[i + 1]]
            for i in range(len(path) - 1)
        )

        print(f"增广路径：{' -> '.join(path)}")
        print(f"瓶颈容量：{bottleneck}\n")

        # 更新正向边、反向边的残量
        for i in range(len(path) - 1):
            u, v = path[i], path[i + 1]
            residual[u][v] -= bottleneck
            residual[v][u] += bottleneck

        max_flow += bottleneck

    return max_flow


edges = [
    ("S", "A", 10),
    ("S", "B", 8),
    ("A", "B", 5),
    ("A", "T", 7),
    ("B", "T", 10),
]

answer = ford_fulkerson(edges, "S", "T")
print(f"最大流：{answer}")

增广路径：S -> B -> T
瓶颈容量：8

增广路径：S -> A -> T
瓶颈容量：7

增广路径：S -> A -> B -> T
瓶颈容量：2

最大流：17


# Gemini code

In [5]:
from collections import deque

def bfs(capacity, source, sink, parent):
    """使用BFS寻找增广路径"""
    visited = [False] * len(capacity)
    queue = deque([source])
    visited[source] = True
    
    while queue:
        u = queue.popleft()
        for v, cap in enumerate(capacity[u]):
            # 如果节点v未访问过，且当前边还有剩余容量 (cap > 0)
            if not visited[v] and cap > 0:
                queue.append(v)
                visited[v] = True
                parent[v] = u # 记录路径
                if v == sink:
                    return True # 找到了一条到汇点的路径
    return False

def edmonds_karp(capacity, source, sink):
    """Edmonds-Karp 计算最大流"""
    n = len(capacity)
    parent = [-1] * n
    max_flow = 0
    
    # 只要还能找到增广路径
    while bfs(capacity, source, sink, parent):
        # 1. 找出这条增广路径上的"瓶颈"（最小剩余容量）
        path_flow = float('inf')
        s = sink
        while s != source:
            path_flow = min(path_flow, capacity[parent[s]][s])
            s = parent[s]
            
        # 2. 更新残量网络（正向减，反向加）
        v = sink
        while v != source:
            u = parent[v]
            capacity[u][v] -= path_flow # 正向边容量减少
            capacity[v][u] += path_flow # 反向边容量增加（允许反悔）
            v = parent[v]
            
        max_flow += path_flow
        
    return max_flow

# ================= 示例运行 =================
# 节点映射: S:0, A:1, B:2, C:3, D:4, T:5
# 构建容量矩阵，capacity[u][v] 表示 u 到 v 的容量
V = 6
capacity_matrix = [[0] * V for _ in range(V)]

edges = [
    (0, 1, 16), (0, 3, 13), # S -> A, S -> C
    (1, 2, 12),             # A -> B
    (3, 1, 4),  (3, 4, 14), # C -> A, C -> D
    (2, 3, 9),  (2, 5, 20), # B -> C, B -> T
    (4, 2, 7),  (4, 5, 4)   # D -> B, D -> T
]

for u, v, w in edges:
    capacity_matrix[u][v] = w

result = edmonds_karp(capacity_matrix, source=0, sink=5)
print(f"该网络的最大流为: {result}")

该网络的最大流为: 23


In [6]:
from collections import deque
import copy

def bfs_augment(residual_capacity, source, sink, parent):
    """用于最大流：在残量网络中寻找增广路径"""
    visited = [False] * len(residual_capacity)
    queue = deque([source])
    visited[source] = True
    
    while queue:
        u = queue.popleft()
        for v, cap in enumerate(residual_capacity[u]):
            if not visited[v] and cap > 0:
                queue.append(v)
                visited[v] = True
                parent[v] = u
                if v == sink:
                    return True
    return False

def find_reachable_nodes(residual_capacity, source):
    """用于最小割：在最终的残量网络中，找出源点能到达的所有节点 (集合A)"""
    visited = [False] * len(residual_capacity)
    queue = deque([source])
    visited[source] = True
    reachable = {source}
    
    while queue:
        u = queue.popleft()
        for v, cap in enumerate(residual_capacity[u]):
            if not visited[v] and cap > 0:
                visited[v] = True
                reachable.add(v)
                queue.append(v)
                
    return reachable

def min_cut(original_capacity, source, sink):
    """计算最大流，并找出最小割的边"""
    n = len(original_capacity)
    # 创建残量网络矩阵（深拷贝原矩阵）
    residual_capacity = copy.deepcopy(original_capacity)
    parent = [-1] * n
    max_flow = 0
    
    # 步骤1：计算最大流
    while bfs_augment(residual_capacity, source, sink, parent):
        path_flow = float('inf')
        s = sink
        while s != source:
            path_flow = min(path_flow, residual_capacity[parent[s]][s])
            s = parent[s]
            
        v = sink
        while v != source:
            u = parent[v]
            residual_capacity[u][v] -= path_flow
            residual_capacity[v][u] += path_flow
            v = parent[v]
            
        max_flow += path_flow
        
    # 步骤2 & 3：找出集合 A
    reachable_A = find_reachable_nodes(residual_capacity, source)
    
    # 步骤4：找出割边（从集合A指向集合B的原图的边）
    cut_edges = []
    for u in reachable_A:
        for v in range(n):
            # v 不在集合A中 (说明在集合B中)，并且原图中有边
            if v not in reachable_A and original_capacity[u][v] > 0:
                cut_edges.append((u, v))
                
    return max_flow, cut_edges, reachable_A

# ================= 示例运行 =================
# 节点: 0(S), 1(A), 2(B), 3(C), 4(D), 5(T)
V = 6
cap_matrix = [[0] * V for _ in range(V)]
edges = [
    (0, 1, 16), (0, 3, 13), 
    (1, 2, 12), 
    (3, 1, 4),  (3, 4, 14), 
    (2, 3, 9),  (2, 5, 20), 
    (4, 2, 7),  (4, 5, 4)   
]
for u, v, w in edges:
    cap_matrix[u][v] = w

flow, cuts, group_A = min_cut(cap_matrix, 0, 5)

print(f"最大流量: {flow}")
print(f"节点集合 A (属于源点侧): {group_A}")
print(f"节点集合 B (属于汇点侧): {set(range(V)) - group_A}")
print(f"需要切断的'最小割'边: {cuts}")

最大流量: 23
节点集合 A (属于源点侧): {0, 1, 3, 4}
节点集合 B (属于汇点侧): {2, 5}
需要切断的'最小割'边: [(1, 2), (4, 2), (4, 5)]
